In [22]:
import random

cities = 4
ants = 2
iterations = 50
evaporation_rate = 0.5  
alpha = 1               
beta = 2                    

# 1. Distance Matrix
distances = [
    [0,  10, 15, 20],
    [10, 0,  35, 25],
    [15, 35, 0,  30],
    [20, 25, 30, 0]
]

# 2. Initial Pheromone concentrations
pheromones = [
    [1, 1, 1, 1],
    [1, 1, 1, 1],
    [1, 1, 1, 1],
    [1, 1, 1, 1]
]

# --- MAIN LOOP ---
best_path = []
best_distance = float('inf') # Infinity

for _ in range(iterations):
    
    # Store all paths taken by ants in this iteration
    all_ant_paths = [] 
    
    # --- ANT MOVEMENT LOOP ---
    for ant in range(ants):
        current_city = 0 # All ants start at city 0 for simplicity
        path = [current_city]
        visited = {current_city}
        
        # Walk until all cities visited
        for _ in range(cities - 1):
            probabilities = []
            possible_next_cities = []
            
            # Calculate probability for each neighbor
            for city in range(cities):
                if city not in visited:
                    # Get Pheromone amount (Tau)
                    tau = pheromones[current_city][city] ** alpha
                    
                    # Get Distance attractiveness (Eta = 1 / distance)
                    dist = distances[current_city][city]
                    eta = (1.0 / dist) ** beta
                    
                    # Score = Pheromone * Distance_Attractiveness
                    prob = tau * eta
                    
                    probabilities.append(prob)
                    possible_next_cities.append(city)
            
            # This is where the 'randomness' happens based on weights
            next_city = random.choices(possible_next_cities, weights=probabilities)[0]
            
            path.append(next_city)
            visited.add(next_city)
            current_city = next_city
        
        # Return to start to complete the loop
        path.append(path[0]) 
        all_ant_paths.append(path)

        # Calculate path distance
        path_dist = 0
        for i in range(len(path) - 1):
            u = path[i]
            v = path[i+1]
            path_dist += distances[u][v]
        
        # Check if this is the best path found so far
        if path_dist < best_distance:
            best_distance = path_dist
            best_path = path
            print(f"Best Path Found at {_} iteration: {best_path}")
            print(f"Best Distance at {_} iteration: {best_distance}")

    # --- PHEROMONE UPDATE LOOP ---
    
    # 1. Evaporation
    for i in range(cities):
        for j in range(cities):
            pheromones[i][j] *= (1.0 - evaporation_rate)
            
    # 2. Deposit phermones
    for path in all_ant_paths:
        
        # Calculate distance of this specific ant's path
        dist = 0
        for i in range(len(path) - 1):
            dist += distances[path[i]][path[i+1]]         
        # Amount to deposit = 1 / distance
        deposit_amount = 1.0 / dist 
        
        # Add to the edges used
        for i in range(len(path) - 1):
            u = path[i]
            v = path[i+1]
            pheromones[u][v] += deposit_amount
            pheromones[v][u] += deposit_amount

# # --- RESULTS ---
# print(f"Best Path Found: {best_path}")
# print(f"Best Distance: {best_distance}")
print("\nFinal Pheromone Matrix (Notice strong values on the best path):")
for row in pheromones:
    print([round(x, 2) for x in row])

Best Path Found at 2 iteration: [0, 1, 2, 3, 0]
Best Distance at 2 iteration: 95
Best Path Found at 2 iteration: [0, 1, 3, 2, 0]
Best Distance at 2 iteration: 80

Final Pheromone Matrix (Notice strong values on the best path):
[0.0, 0.05, 0.05, 0.0]
[0.05, 0.0, 0.0, 0.05]
[0.05, 0.0, 0.0, 0.05]
[0.0, 0.05, 0.05, 0.0]


In [24]:
import numpy as np
np.int = int
from sko.ACA import ACA_TSP

# 1. Setup the Distance Matrix
# Note: scikit-opt works best with numpy arrays
distances = np.array([
    [0,  10, 15, 20],
    [10, 0,  35, 25],
    [15, 35, 0,  30],
    [20, 25, 30, 0]
])

# 2. Define the Objective Function
# This function calculates the total distance of a specific path (routine)
def cal_total_distance(routine):
    num_points, = routine.shape
    sum_distance = 0
    for i in range(num_points):
        # Add distance from current city to next city
        # Use % to wrap around back to the start (TSP loop)
        current_city = routine[i]
        next_city = routine[(i + 1) % num_points]
        sum_distance += distances[current_city, next_city]
    return sum_distance

# 3. Configure ACO (ACA_TSP)
# We map your variables to the library's parameters:
# size_pop = ants
# max_iter = iterations
# alpha, beta = alpha, beta
# rho = evaporation_rate (In this library, rho is the evaporation rate)

aca = ACA_TSP(
    func=cal_total_distance, 
    n_dim=4,                # Number of cities
    size_pop=2,             # Number of ants
    max_iter=50,            # Number of iterations
    distance_matrix=distances, 
    alpha=1, 
    beta=2, 
    rho=0.5                 # Evaporation rate
)

# 4. Run the Algorithm
best_x, best_y = aca.run()

# --- RESULTS ---
print("Optimization Finished!")
print(f"Best Path Found (Indices): {best_x}")
print(f"Best Total Distance: {best_y}")

Optimization Finished!
Best Path Found (Indices): [0 1 3 2]
Best Total Distance: 80
